In [176]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [177]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [178]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [179]:
## check for missing values
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [ ]:
## convert string columns that should be nums to nums
def to_int(val):
    try:
        val = float(val)
        return val
    except:
        return None
df['TotalCharges'] = df['TotalCharges'].apply(to_int)

In [181]:
## check if the column was converted to type float
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

In [182]:
## distribution of predictions
print(df['Churn'].value_counts())
"""
Will have to check that it doesnt guess majority "No" since that is the majority class
i will have ot use PR-AUC to measure how good the classification model is at the end
"""

Churn
No     5174
Yes    1869
Name: count, dtype: int64


'\nWill have to check that it doesnt guess majority "No" since that is the majority class\ni will have ot use PR-AUC to measure how good the classification model is at the end\n'

In [183]:
## check if TotalCharges has missing values
missing_corr = df.isna().corr()
print(missing_corr)
## no correlation so ill just drop the missing rows since its only about ~0.16% of the data

                  customerID  gender  SeniorCitizen  Partner  Dependents  \
customerID               NaN     NaN            NaN      NaN         NaN   
gender                   NaN     NaN            NaN      NaN         NaN   
SeniorCitizen            NaN     NaN            NaN      NaN         NaN   
Partner                  NaN     NaN            NaN      NaN         NaN   
Dependents               NaN     NaN            NaN      NaN         NaN   
tenure                   NaN     NaN            NaN      NaN         NaN   
PhoneService             NaN     NaN            NaN      NaN         NaN   
MultipleLines            NaN     NaN            NaN      NaN         NaN   
InternetService          NaN     NaN            NaN      NaN         NaN   
OnlineSecurity           NaN     NaN            NaN      NaN         NaN   
OnlineBackup             NaN     NaN            NaN      NaN         NaN   
DeviceProtection         NaN     NaN            NaN      NaN         NaN   
TechSupport 

In [184]:
df.dropna(inplace=True)

In [185]:
## nulls are all gone
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [186]:
df = df.drop(columns=['customerID'])

In [187]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [188]:
## convert all categorical features into a class
binary_cols = []
multi_cols = []

for col in df.select_dtypes(include='str').columns:
    n_unique = df[col].nunique()
    if n_unique == 2:
        binary_cols.append(col)
    elif n_unique > 2:
        multi_cols.append(col)


## for binary values in columns
for col in binary_cols:
    vals = sorted(df[col].unique())
    mapping = {vals[0]:0,vals[1]:1}
    df[col] = df[col].map(mapping)

## for multiple categories in a column

df = pd.get_dummies(df,columns=multi_cols,drop_first=True)
df[df.columns[df.dtypes == 'bool']] = df.select_dtypes('bool').astype(int)

In [189]:
## question to answer is binary (to churn or not to churn)
# employ logistic regression, decision tree, and randomforest

In [190]:
from sklearn.model_selection import train_test_split

In [191]:
Y = df['Churn']
X = df.drop(columns='Churn')
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)

In [192]:
## decision tree
from sklearn import tree

In [193]:
d_tree = tree.DecisionTreeClassifier(class_weight = 'balanced')
d_tree = d_tree.fit(x_train,y_train)

In [194]:
y_pred = d_tree.predict(x_test)

In [195]:
len(y_pred) == len(y_test)

True

In [196]:
## metrics precision adn recall

def confusion_matrix(y_pred,y_test):
    tp,fp,tn,fn = 0,0,0,0
    for pred,true in zip(y_pred,y_test):
        if pred == 1 and true == 1:
            tp+=1
        if pred == 0 and true == 1:
            fn+=1
        if pred == 1 and true == 0:
            fp+=1
        if pred == 0 and true == 0:
            tn+=1
    return tp,fp,tn,fn

def metrics(tp,fp,tn,fn):
    precision = tp/(tp+fp)
    recall = tp/(tp+fn)
    accuracy = (tp+tn) / (tp+fp+tn+fn)
    f1 = 2 * (precision * recall) / (precision + recall)
    return precision,recall,accuracy,f1


In [197]:
tp,fp,tn,fn = confusion_matrix(y_pred,y_test)

In [198]:
precision,recall,accuracy,f1 = metrics(tp,fp,tn,fn)

In [199]:
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"Accuracy: {accuracy}")
print(f"F1 : {f1}")

Precision: 0.496
Recall: 0.49732620320855614
Accuracy: 0.7320540156361052
F1 : 0.49666221628838453


In [200]:
from sklearn.ensemble import RandomForestClassifier

In [201]:
rf_classifier = RandomForestClassifier(random_state=0)
rf_classifier.fit(x_train,y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",0
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstra

In [202]:
y_pred = rf_classifier.predict(x_test)

In [203]:
tp,fp,tn,fn = confusion_matrix(y_pred,y_test)

In [204]:
precision = tp/(tp+fp)
recall = tp/(tp+fn)
accuracy = (tp+tn) / (tp+fp+tn+fn)
f1 = 2 * (precision * recall) / (precision + recall)

In [205]:
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"Accuracy: {accuracy}")
print(f"F1 : {f1}")

Precision: 0.636986301369863
Recall: 0.49732620320855614
Accuracy: 0.7910447761194029
F1 : 0.5585585585585586


In [ ]:
## now do a 5-fold CV train/test split